In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import fiona
import geobr
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import os
import re
import time
import unicodedata
import json
from ipyleaflet import Map, GeoJSON, Marker, Popup, LayersControl, FullScreenControl, WidgetControl, AwesomeIcon, Circle, basemaps, ScaleControl
from ipywidgets import HTML, widgets, HBox, Layout
from IPython.display import display
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURAÇÕES
# ============================================
URL = "https://zinecultural.com/blog/comida-di-buteco-juiz-de-fora"
NOME_ARQUIVO = "butecos_comida_2026.csv"

# ============================================
# FUNÇÕES DE PROCESSAMENTO
# ============================================

def limpar_endereco(endereco):
    """Remove o município do endereço"""
    if not endereco:
        return endereco
    endereco_limpo = re.sub(r',\s*Juiz de Fora(\s*-\s*MG)?', '', endereco)
    endereco_limpo = re.sub(r'\s*[-–]\s*Juiz de Fora(\s*-\s*MG)?', '', endereco_limpo)
    return endereco_limpo.strip()

def extrair_campo(texto, padroes):
    """Extrai um campo usando múltiplos padrões de regex"""
    for padrao in padroes:
        match = re.search(padrao, texto, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return ''

def extrair_butecos_completo(url):
    """Extração dos dados via scraping (fallback)"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*',
        'Accept-Language': 'pt-BR,pt;q=0.9,en;q=0.8',
    }
    
    try:
        response = requests.get(url, timeout=15, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"Erro ao acessar: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.content, 'html.parser')
    secao_butecos = None
    
    for h2 in soup.find_all('h2'):
        if 'Participantes Comida di Buteco 2026' in h2.get_text():
            secao_butecos = h2
            break
    
    if not secao_butecos:
        print("Seção não encontrada!")
        return pd.DataFrame()

    dados_butecos = []
    current = secao_butecos.find_next_sibling()
    
    while current:
        if current.name == 'h3':
            texto_h3 = current.get_text(strip=True)
            if re.search(r'\d+\s*de\s*40', texto_h3):
                nome = re.sub(r'^\d+\s*de\s*40\s*[–-]\s*', '', texto_h3).strip()
                
                petisco = descricao = endereco = contato = instagram = funcionamento = ''
                next_elem = current.find_next_sibling()
                
                while next_elem and next_elem.name != 'h3':
                    texto_completo = next_elem.get_text('', strip=True)
                    
                    if not petisco:
                        petisco = extrair_campo(texto_completo, [r'(?:🥘\s*)?Petisco:\s*(.+?)'])
                    if not descricao:
                        descricao = extrair_campo(texto_completo, [r'(?:🥣\s*)?Descrição:\s*(.+?)'])
                    if not endereco:
                        endereco = extrair_campo(texto_completo, [r'(?:🏡\s*)?<strong>Endereço:</strong>\s*(.+?)'])
                        if endereco:
                            endereco = re.sub(r'<[^>]+>', '', endereco)
                            endereco = re.sub(r'[🏡]', '', endereco).strip()
                            endereco = limpar_endereco(endereco)
                    if not contato:
                        contato = extrair_campo(texto_completo, [r'(?:☎\s*)?Contato:\s*(.+?)'])
                    if not instagram:
                        instagram = extrair_campo(texto_completo, [r'(?:📱\s*)?Instagram:\s*@?([^\s]+)'])
                    if not funcionamento:
                        funcionamento = extrair_campo(texto_completo, [r'(?:⏰\s*)?Funcionamento:\s*(.+)'])
                    
                    next_elem = next_elem.find_next_sibling()
                
                dados_butecos.append({
                    'Name': nome, 'Endereço': endereco, 'Petisco': petisco,
                    'Contato': contato, 'Instagram': instagram,
                    'Descrição': descricao, 'Funcionamento': funcionamento
                })
        current = current.find_next_sibling()
        if current and current.name == 'h2':
            break
    
    return pd.DataFrame(dados_butecos)

def salvar_csv_com_backup(df, nome_arquivo):
    """Salva CSV com backup automático"""
    if os.path.exists(nome_arquivo):
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        nome_backup = f"{Path(nome_arquivo).stem}_backup_{timestamp}{Path(nome_arquivo).suffix}"
        try:
            os.rename(nome_arquivo, nome_backup)
            print(f"Backup criado: {nome_backup}")
        except Exception as e:
            print(f"Não foi possível criar backup: {e}")
    df.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')
    print(f"Arquivo salvo: {nome_arquivo}")

def normalizar_texto(texto):
    """Normaliza texto para comparação"""
    if pd.isna(texto):
        return texto
    texto = str(texto)
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')
    return texto.strip().upper()

def limpar_nome_buteco(nome):
    """Limpa nomes de butecos"""
    if pd.isna(nome):
        return nome
    nome = str(nome)
    padroes_remover = ['- Juíz de Fora', '- Bar e Restaurante Alto dos Passos', ' Endereço:']
    for padrao in padroes_remover:
        nome = nome.replace(padrao, '')
    return nome.strip()

def processar_dados(gdf_com_endereco):
    """Processa e mescla todos os dados"""
    gdf_final = gdf_com_endereco.copy()
    
    # Colunas auxiliares
    gdf_final['Name_clean'] = gdf_final['Name'].apply(limpar_nome_buteco)
    gdf_final['Name_norm'] = gdf_final['Name_clean'].apply(normalizar_texto)
    
    # Preparar endereços limpos
    enderecos_limpo = gdf_com_endereco.copy()
    enderecos_limpo['Name_clean'] = enderecos_limpo['Name'].apply(limpar_nome_buteco)
    enderecos_limpo['Name_norm'] = enderecos_limpo['Name_clean'].apply(normalizar_texto)
    
    # Mapeamento manual
    mapeamento_nomes = {
        'Bar do Antonio': 'Bar do Antônio', 'Bar Torresmo': 'Bar do Torresmo',
        'Bar Du Buneco': 'Bar Du Buneco- Juíz de Fora', 'Bar Batata D\'Mola': "Batata d' Mola",
        'Birosca Bar e Restaurante': 'Birosca', 'Lero Lero': 'Lero Lero Bar',
        'Pão Moiado Bar': 'Pão Moiado', 'Ibitibar': 'IBITIBAR',
        'Carlota': 'Carlota Bar e Restaurante', 'Coliseum Bar': 'Coliseum Bar e Restaurante',
        'Informal Bar& Restaurante': 'Informal Bar & Restaurante',
        'Compadre Grill Costelaria': 'Costelaria Compadre Grill',
        'Don Juan Gastronomia e eventos': 'Don Juan Gastronomia',
        'Espetinho da Villa': 'Espetinho Da Villa- Bar e Restaurante Alto dos Passos',
    }
    
    # Dicionário para acesso rápido
    enderecos_dict = {}
    for _, row in enderecos_limpo.iterrows():
        enderecos_dict[row['Name_clean']] = row
        enderecos_dict[row['Name_norm']] = row
    
    # Adicionar mapeamento manual
    for nome_gdf, nome_endereco in mapeamento_nomes.items():
        nome_gdf_clean = limpar_nome_buteco(nome_gdf)
        if nome_gdf_clean not in enderecos_dict:
            nome_end_clean = limpar_nome_buteco(nome_endereco)
            if nome_end_clean in enderecos_dict:
                enderecos_dict[nome_gdf_clean] = enderecos_dict[nome_end_clean]
    
    # Preencher dados
    for idx, row in gdf_final.iterrows():
        nome_clean = row['Name_clean']
        nome_norm = row['Name_norm']
        
        registro = None
        if nome_clean in enderecos_dict:
            registro = enderecos_dict[nome_clean]
        elif nome_norm in enderecos_dict:
            registro = enderecos_dict[nome_norm]
        
        if registro is not None:
            gdf_final.at[idx, 'Endereço'] = registro['Endereço']
            gdf_final.at[idx, 'Petisco'] = registro['Petisco']
            gdf_final.at[idx, 'Contato'] = registro['Contato']
            gdf_final.at[idx, 'Instagram'] = registro['Instagram']
            gdf_final.at[idx, 'Descrição'] = registro['Descrição']
            gdf_final.at[idx, 'Funcionamento'] = registro['Funcionamento']
    
    # Reorganizar colunas
    gdf_final = (gdf_final
                 .drop(columns=['Name', 'Name_clean'])
                 .rename(columns={'Name_norm': 'Name'})
                 .reindex(columns=['Name'] + [col for col in gdf_final.columns if col not in ['Name', 'Name_clean', 'Name_norm']]))
    
    return gdf_final


# ============================================
# CARREGAMENTO OTIMIZADO
# ============================================

@st.cache_data(ttl=3600)
def carregar_todos_os_dados():
    """Carrega todos os dados uma vez e cacheia"""
    with st.spinner("⏳ Carregando dados..."):
        try:
            if os.path.exists(NOME_ARQUIVO):
                print("🔄 Carregando dados do arquivo CSV...")
                enderecos = pd.read_csv(NOME_ARQUIVO)
                print(f"✅ Dados carregados do CSV: {len(enderecos)} registros")
            else:
                print("⚠️ Arquivo CSV não encontrado. Fazendo scraping...")
                df_butecos = extrair_butecos_completo(URL)
                if len(df_butecos) > 0:
                    salvar_csv_com_backup(df_butecos, NOME_ARQUIVO)
                enderecos = df_butecos
            
            if len(enderecos) < 30:
                st.error("⚠️ Poucos dados encontrados. Tentando refazer scraping...")
                enderecos = extrair_butecos_completo(URL)
                salvar_csv_com_backup(enderecos, NOME_ARQUIVO)
            
            coordenadas = fiona.listlayers("bares_jf.kml")
            gdfs = []
            for layer in coordenadas:
                gdf_layer = gpd.read_file("bares_jf.kml", layer=layer)
                gdf_layer['camada_origem'] = layer
                gdfs.append(gdf_layer)
            gdf = pd.concat(gdfs, ignore_index=True)
            
            gdf_com_endereco = gdf.merge(enderecos, on='Name', how='left')
            gdf_final = processar_dados(gdf_com_endereco)
            
            return gdf_final, enderecos
            
        except Exception as e:
            st.error(f"❌ Erro ao carregar dados: {str(e)}")
            raise

try:
    gdf_final, enderecos = carregar_todos_os_dados()
except Exception:
    st.error("❌ Não foi possível carregar os dados. Verifique se os arquivos estão presentes.")
    st.stop()

# ============================================
# DEFININDO O POLÍGONO E MAPA
# ============================================

jf = geobr.read_municipality(code_muni=3136702, year=2022)

if jf is None or jf.empty:
    st.error("❌ Não foi possível carregar o polígono do município.")
    st.stop()

jf_wgs84 = jf.to_crs(epsg=4326)
jf_bounds = jf_wgs84.total_bounds
center_lat = (jf_bounds[1] + jf_bounds[3]) / 2
center_lon = (jf_bounds[0] + jf_bounds[2]) / 2
map_bounds = [[jf_bounds[1], jf_bounds[0]], [jf_bounds[3], jf_bounds[2]]]

def calculate_zoom(bounds):
    lat_diff = bounds[3] - bounds[1]
    lon_diff = bounds[2] - bounds[0]
    max_diff = max(lat_diff, lon_diff)
    
    if max_diff > 5: return 0
    elif max_diff > 2: return 1
    elif max_diff > 1: return 2
    elif max_diff > 0.5: return 3
    elif max_diff > 0.2: return 4
    elif max_diff > 0.1: return 5
    else: return 6

initial_zoom = calculate_zoom(jf_bounds)

# Cria o mapa ipyleaflet
m = Map(
    basemap=basemaps.Esri.WorldImagery,
    center=(float(center_lat), float(center_lon)),
    zoom=int(initial_zoom),
    scroll_wheel_zoom=True, dragging=True, touch_zoom=True,
    double_click_zoom=True, box_zoom=True, keyboard=True,
    attribution_control=False
)

# Adiciona polígono municipal
jf_geojson = jf.to_crs(epsg=4326)
jf_geojson_data = json.loads(jf_geojson.to_json())

geojson_style = {
    'fillColor': 'lightblue',
    'color': '#FEBF57',
    'weight': 2,
    'fillOpacity': 0.3
}

geojson_layer = GeoJSON(
    data=jf_geojson_data,
    style=geojson_style,
    name='Limite Municipal'
)

m.add_layer(geojson_layer)

beer_icon = AwesomeIcon(name='beer', marker_color='orange', icon_color='white', prefix='fa')
bars_data = {}
highlight_circles = {}

for idx, row in gdf_final.iterrows():
    if row['geometry'] is None:
        continue
    
    if row['geometry'].geom_type == 'Point':
        lon, lat = row['geometry'].x, row['geometry'].y
    elif row['geometry'].geom_type in ['Polygon', 'MultiPolygon']:
        centroid = row['geometry'].centroid
        lon, lat = centroid.x, centroid.y
    else:
        continue
    
    lat = float(lat)
    lon = float(lon)
    
    nome = row['Name'] if row['Name'] else "Sem nome"
    endereco = row['Endereço'] if row['Endereço'] else "Endereço não disponível"
    
    bars_data[nome] = {'lat': lat, 'lon': lon, 'Endereço': endereco}
    
    popup_content = widgets.HTML(
        value=f"<b>{nome}</b><br><i>{endereco}</i>",
        layout=widgets.Layout(min_width='200px')
    )
    
    marker = Marker(location=(lat, lon), icon=beer_icon, title=nome, draggable=False)
    popup = Popup(child=popup_content, close_button=True, auto_close=True)
    marker.popup = popup
    
    m.add_layer(marker)

# Controles
valid_bars = sorted([nome for nome in bars_data.keys() if nome != "Sem nome"])

btn_reset = widgets.Button(
    description="Centralizar", button_style='warning', icon='refresh',
    tooltip='Voltar para visão geral do município',
    layout=widgets.Layout(width='auto', min_width='100px', padding='4px 8px')
)

def reset_map(b=None):
    for circle in list(highlight_circles.values()):
        m.remove_layer(circle)
    highlight_circles.clear()
    bar_dropdown.value = '🍺 Selecione um bar...'
    m.fit_bounds(map_bounds)

btn_reset.on_click(reset_map)

reset_container = widgets.HBox(
    [btn_reset],
    layout=widgets.Layout(
        background='white', padding='6px 10px', border_radius='4px',
        box_shadow='0 1px 5px rgba(0,0,0,0.4)'
    )
)

bar_dropdown = widgets.Dropdown(
    options=['🍺 Selecione um bar...'] + valid_bars,
    value='🍺 Selecione um bar...',
    layout=widgets.Layout(width='220px')
)

def on_bar_selected(change):
    selected_bar = change['new']
    
    for circle in list(highlight_circles.values()):
        m.remove_layer(circle)
    highlight_circles.clear()
    
    if selected_bar and selected_bar != '🍺 Selecione um bar...' and selected_bar in bars_data:
        coords = bars_data[selected_bar]
        m.center = (coords['lat'], coords['lon'])
        m.zoom = 16
        
        highlight_circle = Circle(
            location=(coords['lat'], coords['lon']),
            radius=80, color='red', weight=3, fill=False, interactive=False
        )
        m.add_layer(highlight_circle)
        highlight_circles[selected_bar] = highlight_circle

bar_dropdown.observe(on_bar_selected, names='value')

dropdown_label = widgets.HTML(
    value='<b style="font-size:13px;margin-right:5px;">Bar:</b>',
    layout=widgets.Layout(display='inline-block')
)

dropdown_container = widgets.HBox(
    [dropdown_label, bar_dropdown],
    layout=widgets.Layout(
        align_items='center', background='white', padding='8px 10px',
        border_radius='4px', box_shadow='0 1px 5px rgba(0,0,0,0.4)',
        min_width='250px'
    )
)

m.add_control(WidgetControl(widget=dropdown_container, position='topleft'))
m.add_control(WidgetControl(widget=reset_container, position='bottomright'))
m.add_control(ScaleControl(position='bottomleft'))
m.add_control(LayersControl(position='topright', collapsed=False))
m.add_control(FullScreenControl())

display(m)